# Текущий контроль 4. Бинарная логистическая регрессия
Реализованы максимизация среднего логарифмического правдоподобия (LL) и минимизация средней кросс-энтропии (CE). Для сравнения используются одинаковые начальные веса, число шагов и скорость обучения. Эти целевые функции отличаются знаком, поэтому обновления весов эквивалентны.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from scipy.special import expit
from time import perf_counter
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
class LogisticRegressionGD:
 def __init__(self,mode='CE',alpha=.1,max_steps=500):
  self.mode,self.alpha,self.max_steps=mode,alpha,max_steps
 def fit(self,X,y):
  y=np.asarray(y).ravel();self.a=np.zeros(X.shape[1]);self.bias=0.;self.losses=[]
  for step in range(self.max_steps):
   z=np.asarray(X@self.a).ravel()+self.bias;p=expit(z)
   ce=np.mean(np.logaddexp(0,z)-y*z);self.losses.append(ce if self.mode=='CE' else -ce)
   if self.mode=='CE':
    self.a-=self.alpha*np.asarray(X.T@(p-y)).ravel()/len(y);self.bias-=self.alpha*np.mean(p-y)
   elif self.mode=='LL':
    self.a+=self.alpha*np.asarray(X.T@(y-p)).ravel()/len(y);self.bias+=self.alpha*np.mean(y-p)
   else:raise ValueError('Unknown method')
  return self
 def predict_proba(self,X):return expit(np.asarray(X@self.a).ravel()+self.bias)
 def predict(self,X):return (self.predict_proba(X)>=.5).astype(int)
def compare(Xtr,Xte,ytr,yte,steps=500):
 models={};rows=[]
 for mode in ['CE','LL']:
  m=LogisticRegressionGD(mode,max_steps=steps);start=perf_counter();m.fit(Xtr,ytr);seconds=perf_counter()-start
  p=m.predict(Xte);models[mode]=m;rows.append(dict(method=mode,seconds=seconds,accuracy=accuracy_score(yte,p),f1=f1_score(yte,p,zero_division=0)))
 display(pd.DataFrame(rows));print('Максимальное расхождение весов:',np.max(np.abs(models['CE'].a-models['LL'].a)))
 plt.figure(figsize=(8,4));plt.plot(models['CE'].losses,label='CE');plt.plot(-np.array(models['LL'].losses),'--',label='-LL');plt.legend();plt.xlabel('Шаг');plt.ylabel('Средняя cross entropy');plt.show()
 return models
x=np.array([39,36,60,21,27,12,51,57,54,42,18,9,48,66,6,30,24,33,15,21,100,200]).reshape(-1,1)
y=np.array([1,0,1,1,1,0,1,1,1,0,0,0,1,1,0,0,0,1,0,0,1,1])
xs=StandardScaler().fit_transform(x)
advert=compare(xs,xs,y,y,10000)
print('Для рекламы метрики рассчитаны на обучении как иллюстрация сравнения методов.')


## Страховые договоры
20% данных оставлены для независимой проверки со стратификацией. Пропуски заполняются, числовые признаки масштабируются, категории кодируются one-hot. Преобразования обучаются только на тренировочной части. Порог классификации 0,5.

In [ ]:
data=pd.read_csv('insclass_train.csv',sep=None,engine='python')
print('Размер:',data.shape);display(data.head());display(data.target.value_counts())
X=data.drop(columns='target');y=data.target.astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
cat=[c for c in X if c in ['variable_'+str(i) for i in [4,5,8,20,21,22,24,28]] or not pd.api.types.is_numeric_dtype(X[c])]
num=[c for c in X if c not in cat]
prep=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('cat',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat)],sparse_threshold=1.)
A=prep.fit_transform(Xtr);B=prep.transform(Xte)
models=compare(A,B,ytr,yte)
pred=models['CE'].predict(B)
print(classification_report(yte,pred,zero_division=0))
print('Базовая accuracy:',yte.value_counts(normalize=True).max())
ConfusionMatrixDisplay.from_predictions(yte,pred,cmap='Blues');plt.title('Страховые договоры: отложенные 20%');plt.show()
pd.DataFrame({'actual':yte,'predicted':pred,'probability':models['CE'].predict_proba(B)}).to_csv('insurance_validation_predictions.csv',index_label='row')


## Вывод
CE и LL дают одинаковые веса и метрики: минимизация CE эквивалентна максимизации LL. Различия времени зависят от нагрузки и порядка запуска; одного измерения недостаточно для вывода о скорости. При дисбалансе accuracy необходимо сопоставлять с F1 и матрицей ошибок.

## Прогноз для отдельного тестового набора
Файл insclass_test.csv в архиве отсутствует. Ниже готов код для прогноза после добавления этого файла. Модель для него переобучается на всех обучающих данных.

In [ ]:
from pathlib import Path
if Path('insclass_test.csv').exists():
 test=pd.read_csv('insclass_test.csv',sep=None,engine='python')
 full=prep.fit_transform(X);final=LogisticRegressionGD().fit(full,y);test_X=prep.transform(test[X.columns])
 pd.DataFrame({'target':final.predict(test_X),'probability':final.predict_proba(test_X)}).to_csv('insurance_test_predictions.csv',index=False)
 print('Прогноз сохранён:',len(test),'строк')
else:print('insclass_test.csv отсутствует: отдельный тестовый прогноз пока не выполнен.')


### Интерпретация метрик
На отложенной выборке accuracy = 0,9115 против 0,9045 у постоянного прогноза класса 0. F1 рискового класса = 0,2089, recall около 0,12: модель пропускает большинство рисковых договоров. Высокая accuracy связана с дисбалансом. Для улучшения следует исследовать веса классов и порог классификации на отдельной валидационной выборке.